← [Overview](00_overview.ipynb)

# Representation

Clustering decides **which** periods group together. Representation decides **what each
group's single profile looks like**. In tsam these are separate, recombinable steps: every
clustering method sets a default representation, and you can override it freely — Ward with a
maxoid, k-means with a medoid, any pairing you like.

A representation rule acts on **one cluster at a time**, independently of every other cluster
and of whatever method formed it. So one cluster is the whole story, and that is what this
notebook follows.

| | |
|---|---|
| **In** | the member periods of one formed cluster — a `n_members × (n_attributes · n_timesteps)` matrix |
| **Inside** | one of six rules for condensing those members into a single profile |
| **Out** | one profile: `n_attributes · n_timesteps` numbers — the same shape as any member |

This notebook works each rule by hand, to show *how* it arrives at its profile. To see the six
compared on the same data — which keeps the peak, which keeps the mean, which is a real day —
see [Comparing representations](../../tutorials/comparing_representations.ipynb).

## 1  What comes in

A **formed cluster**: some set of periods that a clustering method decided belong together.
Representation does not care which method that was, or why — only which rows it is handed.

So we simply declare one. The cluster needs enough members for the rules to differ (with two
periods the medoid and maxoid tie, and the mean sits exactly halfway between them), so we take
four days of the [tiny six-day set](01_preprocessing.ipynb) as given. This is exactly the
grouping Ward finds at k=2, but nothing here depends on that — the rules see only the
matrix.

In [ ]:
import numpy as np
import pandas as pd

ATTRS = ["solar", "load"]
N_TIMESTEPS = 4

# The preprocessed period matrix D from 01_preprocessing: normalized to [0, 1] and
# unstacked, one row per day, columns (attribute, timestep).
D = pd.read_csv("../../data/tiny_periods.csv", header=[0, 1], index_col=0)

# The cluster, taken as given. Every rule below sees only these four rows.
MEMBERS = [2, 3, 4, 5]
M = D.loc[MEMBERS].values  # (4 members, 8 = 2 attributes x 4 timesteps)

print("the member matrix M:", M.shape, "— four periods, eight coordinates each")
D.loc[MEMBERS].round(4)

## 2  What has to come out

One profile for the cluster: **one value per attribute per timestep** — the same eight numbers
any member has. Whatever the rule, the output shape is fixed; only the way those eight numbers
are chosen changes.

That leaves one real question, and it is the one the six rules disagree about: **should the
profile be a period that actually happened, or a constructed one?**

| tsam name | The profile is… | Chosen |
|---|---|---|
| `mean` | the members' average at each timestep | per timestep |
| `medoid` | the member closest to its cluster-mates | **per period** |
| `maxoid` | the member farthest from the rest of the data | **per period** |
| `distribution` | values re-sorted to keep the cluster's duration curve | per timestep |
| `distribution_minmax` | the same, but each attribute's min and max kept exact | per timestep |
| `minmax_mean` | per attribute: the members' min, max or mean at each timestep | per timestep |

A rule chosen **per period** lifts one whole member out of the cluster, so every internal
correlation that day had — solar and load at the same hour — survives intact. A rule chosen
**per timestep** assembles each of the eight numbers independently, and the result is a profile
no day ever followed.

## 3  Inside: the six rules

### `medoid` and `maxoid`: pick a real day

Both select an existing member, so both measure distances — but **they do not measure the same
distances**. The medoid looks **inward**: the member with the smallest total distance to its
own cluster-mates. The maxoid looks **outward**: the member with the largest total distance to
*every period in the dataset*, not just its cluster-mates — it is chosen for being extreme
within the whole series, not merely within its group.

That difference in what they measure is the whole reason they return different days.

In [ ]:
def distances(X, Y):
    """Euclidean distance between every row of X and every row of Y."""
    return np.sqrt(((X[:, None, :] - Y[None, :, :]) ** 2).sum(-1))


labels = [f"day{d}" for d in MEMBERS]

within = distances(M, M)
print("Distances among the cluster members (normalized space):")
print(pd.DataFrame(within, index=labels, columns=labels).round(3).to_string())

print("\nmedoid — total distance to the other MEMBERS (smallest wins):")
for i, day in enumerate(MEMBERS):
    print(f"  day{day}: {within[i].sum():.3f}")
print(f"  -> medoid = day{MEMBERS[int(np.argmin(within.sum(axis=0)))]}")

print("\nmaxoid — total distance from ALL SIX days in the series (largest wins):")
outward = distances(D.values, M)
for i, day in enumerate(MEMBERS):
    print(f"  day{day}: {outward[:, i].sum():.3f}")
print(f"  -> maxoid = day{MEMBERS[int(np.argmax(outward.sum(axis=0)))]}")

The medoid is **day4**, the maxoid **day5** — the same cluster, two different real days,
because one rule minimizes an inward distance and the other maximizes an outward one.

### `distribution`: keep the value distribution, not the shape

The distribution rule gives up on matching any member's timing and matches the cluster's
**duration curve** instead — how often each value level occurs. It does this in three moves,
per attribute:

1. **Pool** every value in the cluster — all members, all timesteps (here 4 × 4 = 16 per attribute).
2. **Sort** them and average them down into `n_timesteps` levels: the cluster's duration curve
   at four points.
3. **Order** those levels along the period by ranking the cluster's *mean* profile — the values
   come from the distribution, the ordering is borrowed from the average shape.

Step 2 is why this rule keeps the cluster's **mean** for free: the sorted pool is cut into
equal-sized groups and each is averaged, so the average of the levels is the average of the
pool. It is the only rule besides `mean` with that property.

With `preserve_minmax` the lowest and highest pooled values are written into the first and last
levels verbatim, so the cluster's true extremes survive step 2's averaging — and the mean
guarantee is given up in exchange.

### `minmax_mean`: per attribute, per timestep

Each attribute is handled separately: take the members' `min`, `max` or `mean` at each timestep.
Below, `load` is set to `max` and `solar` to `mean` — an envelope on demand, an average on
supply.

In [ ]:
def rule_mean(members_matrix):
    """Per-timestep average across the members."""
    return members_matrix.mean(axis=0)


def rule_medoid(members_matrix):
    """The member with the smallest total distance to its cluster-mates."""
    total = distances(members_matrix, members_matrix).sum(axis=0)
    return members_matrix[int(np.argmin(total))]


def rule_maxoid(members_matrix, all_periods):
    """The member with the largest total distance to every period in the series."""
    total = distances(all_periods, members_matrix).sum(axis=0)
    return members_matrix[int(np.argmax(total))]


def rule_distribution(members_matrix, preserve_minmax=False):
    """Values from the cluster's pooled duration curve, ordered by the mean profile."""
    n_members = members_matrix.shape[0]
    n_attrs = len(ATTRS)
    per_attr = members_matrix.reshape(n_members, n_attrs, N_TIMESTEPS).transpose(
        1, 0, 2
    )

    # 1. pool and 2. sort into n_timesteps duration-curve levels
    pooled = per_attr.reshape(n_attrs, -1).copy()
    pooled.sort(axis=1, kind="stable")
    levels = pooled.reshape(n_attrs, N_TIMESTEPS, n_members).mean(axis=2)
    if preserve_minmax:
        levels[:, 0] = pooled[:, 0]
        levels[:, -1] = pooled[:, -1]

    # 3. order the levels by the ranking of the cluster's mean profile
    order = np.round(per_attr.mean(axis=1), 10).argsort(axis=1, kind="stable")
    profile = np.empty_like(levels)
    profile[np.arange(n_attrs)[:, None], order] = levels
    return profile.ravel()


def rule_minmax_mean(members_matrix, spec):
    """Per attribute, per timestep: the members' min, max or mean."""
    profile = np.zeros(len(ATTRS) * N_TIMESTEPS)
    for a, attr in enumerate(ATTRS):
        start, end = a * N_TIMESTEPS, (a + 1) * N_TIMESTEPS
        block = members_matrix[:, start:end]
        profile[start:end] = {
            "min": block.min(axis=0),
            "max": block.max(axis=0),
            "mean": block.mean(axis=0),
        }[spec[attr]]
    return profile


representatives = {
    "mean": rule_mean(M),
    "medoid": rule_medoid(M),
    "maxoid": rule_maxoid(M, D.values),
    "distribution": rule_distribution(M),
    "distribution_minmax": rule_distribution(M, preserve_minmax=True),
    "minmax_mean": rule_minmax_mean(M, {"solar": "mean", "load": "max"}),
}

print("Six representatives of the same cluster, normalized:")
pd.DataFrame(representatives, index=D.columns).T.round(4)

## 4  What comes out

Eight numbers, whichever rule ran — the shape contract the rest of the pipeline depends on.
What differs is where those numbers came from, and the two questions below are the ones the
downstream model actually feels.

**Is it a real period?** Only `medoid` and `maxoid` can promise that; the other four assemble
a profile timestep by timestep.

**Does it still carry the cluster's mean?** Only `mean` and `distribution` do. Every other rule
hands on a profile whose weighted total is wrong — which is precisely the gap the optional
[rescaling](05_rescaling.ipynb) step exists to close.

In [ ]:
rows = []
for name, profile in representatives.items():
    match = next(
        (day for day in MEMBERS if np.allclose(D.loc[day].values, profile, atol=1e-9)),
        None,
    )
    keeps_mean = np.allclose(
        profile.reshape(len(ATTRS), N_TIMESTEPS).mean(axis=1),
        M.reshape(len(M), len(ATTRS), N_TIMESTEPS).mean(axis=(0, 2)),
        atol=1e-12,
    )
    rows.append(
        {
            "rule": name,
            "a real period?": f"yes — day{match}"
            if match is not None
            else "constructed",
            "keeps the cluster's means?": "yes" if keeps_mean else "no",
        }
    )

pd.DataFrame(rows).set_index("rule")

Those two columns are the whole design space of this step, and **no rule scores yes on
both**. A profile can be a day that really occurred, or it can carry the group's average — not
both at once, because the average of a group is almost never one of its members.

That is not a defect in tsam's rules; it is a property of summarising many periods with one.
Everything downstream follows from which half you gave up:

* gave up the mean (`medoid`, `maxoid`, `distribution_minmax`, `minmax_mean`)
  → [rescaling](05_rescaling.ipynb) restores the totals afterwards;
* gave up realism (`mean`, `distribution`)
  → [extreme periods](04_extreme_periods.ipynb) injects back the specific real periods you
  could not afford to average away.

---

**Up next:**

* [Extreme periods](04_extreme_periods.ipynb) — averaging destroys peaks; extremes force a
  chosen period into the cluster set so it cannot be averaged away.
* [Rescaling](05_rescaling.ipynb) — how the four mean-breaking rules get their totals back, and
  what that correction costs.

**See also:**

* [Comparing representations](../../tutorials/comparing_representations.ipynb) — the six rules
  side by side on one cluster, with the peaks and paths drawn.
* [Representations how-to](../../how-to/representations.ipynb) — choosing between them on a
  realistic series.
* [Notation and equations](../../reference/notation.md) — every symbol and formula on one page.